# Patent Citation Trend — yearly incoming citations per patent (granted + application)

For every US patent, its **year-by-year incoming citations**, anchored at the **grant year**, split into
**examiner** / **non-examiner** (applicant + other; `cited by third party` excluded), for two citation
sources: **granted** patent->patent (`g_us_patent_citation`) and **application** (pre-grant) citations
(`g_us_application_citation`, cited pgpub mapped to a granted patent via `pg_granted_pgpubs_crosswalk`).

Citing-year = the **citing patent's grant year** for both sources (kept identical so the two trends share the
same `yrs_since_grant` axis; application citations are physically made earlier, at the citing application's
filing, so their `yrs_since_grant` is an upper bound).

## Raw / input data
```
/project/jevans/Dawoon/Science of Science/PatentView/Granted/g_us_patent_citation.tsv.zip        # granted patent->patent
/project/jevans/Dawoon/Science of Science/PatentView/Granted/g_us_application_citation.tsv.zip   # application (pgpub cited)
/project/jevans/Dawoon/Science of Science/PatentView/Pregranted_260605/pg_granted_pgpubs_crosswalk.tsv.zip  # pgpub->patent
/project/jevans/Dawoon/Science of Science/Data/PatentView/Granted/g_patent.tsv.zip                    # grant years
```

## Output
`/project/jevans/Dawoon/Science of Science/PatentView/output/patent_citation_trend.parquet` — `patent_id, grant_year, cite_year, yrs_since_grant,
pat2pat_examiner, pat2pat_non_examiner, app2pat_examiner, app2pat_non_examiner` (one row per
patent x citing-year with >=1 citation from either source).

In [3]:
import os, sys, gc, time
import numpy as np, pandas as pd
sys.path.insert(0, '/project/jevans/Dawoon/Science of Science/PatentView')
import pv_common as pv
ROOT, D, OUT = pv.BASE, pv.GRANTED, pv.OUT
OUT_FP = pv.out('patent_citation_trend.parquet')
XWALK = pv.pregranted('pg_granted_pgpubs_crosswalk.tsv.zip')   # arrived 2026-08-28
pv.preflight('patent_citation_trend')

granted    : /project/jevans/Dawoon/Science of Science/PatentView/Granted   (35 zip files)
pregranted : /project/jevans/Dawoon/Science of Science/PatentView/Pregranted   (25 files)
output     : /project/jevans/Dawoon/Science of Science/PatentView/output

  patent_citation_trend           OK


True

## 1. grant years

In [4]:
%%time
gp = pd.read_csv(os.path.join(D, 'g_patent.tsv.zip'), sep='\t',
                 usecols=['patent_id', 'patent_type', 'patent_date'], dtype={'patent_id': str, 'patent_type': str})
gp = gp[gp['patent_type'] == 'utility']
gp['gy'] = pd.to_datetime(gp['patent_date'], errors='coerce').dt.year
gp = gp.dropna(subset=['gy']); pid2year = dict(zip(gp['patent_id'], gp['gy'].astype(int)))
print(f'utility patents: {len(pid2year):,}'); del gp; gc.collect()

utility patents: 8,531,961
CPU times: user 17.6 s, sys: 940 ms, total: 18.6 s
Wall time: 18.6 s


1054

## 2. Scan patent→patent citations → (cited, citing grant year, examiner/non-examiner)

In [ ]:
%%time
acc = []; t0 = time.time(); seen = 0
for ch in pd.read_csv(os.path.join(D, 'g_us_patent_citation.tsv.zip'), sep='\t',
                      usecols=['patent_id', 'citation_patent_id', 'citation_category'], dtype=str, chunksize=5_000_000):
    ch = ch.dropna(subset=['patent_id', 'citation_patent_id'])
    cat = ch['citation_category'].fillna('').str.lower()
    notp = ~cat.str.contains('third party', na=False)
    ch = ch.loc[notp]; cat = cat.loc[notp]
    cy = ch['patent_id'].map(pid2year); dy = ch['citation_patent_id'].map(pid2year)
    keep = (cy.notna() & dy.notna()).values
    if keep.any():
        diff = (cy.values - dy.values)[keep]
        cited = ch['citation_patent_id'].values[keep]
        _cv = cat.values[keep]
        # three-way, as in patent_citation.ipynb: the empty category is its own bucket,
        # not applicant. Pre-2001 rows carry no origin field at all.
        bucket = np.where(_cv == 'cited by examiner', 'examiner',
                          np.where(_cv == '', 'unknown', 'non_examiner'))
        pos = diff >= 0
        d = pd.DataFrame({'cited': cited[pos], 'cite_year': cy.values[keep][pos].astype(int), 'bucket': bucket[pos]})
        acc.append(d.groupby(['cited', 'cite_year', 'bucket']).size().reset_index(name='n'))
    seen += len(ch)
pat = pd.concat(acc, ignore_index=True).groupby(['cited', 'cite_year', 'bucket'])['n'].sum().reset_index()
print(f'[{time.time()-t0:.0f}s] scanned {seen:,} rows -> {len(pat):,} (cited, year, bucket) groups')

In [ ]:
ch.head()

## 2b. Scan application citations -> (cited, citing grant year, examiner/non-examiner)

Cited pre-grant publication (`citation_document_number` = pgpub_id) mapped to a granted `patent_id` via `pg_granted_pgpubs_crosswalk`; same third-party exclusion and examiner/non-examiner split.

In [4]:
%%time
XW = XWALK
xw = pd.read_csv(XW, sep='\t', usecols=['pgpub_id', 'patent_id'], dtype=str, on_bad_lines='skip')
xw = xw.dropna(subset=['pgpub_id', 'patent_id']).drop_duplicates('pgpub_id')
pgpub2pid = dict(zip(xw['pgpub_id'], xw['patent_id'])); del xw; gc.collect()
print(f'crosswalk pgpub->patent: {len(pgpub2pid):,}')
accA = []; t0 = time.time(); seen = 0
for ch in pd.read_csv(os.path.join(D, 'g_us_application_citation.tsv.zip'), sep='\t',
                      usecols=['patent_id', 'citation_document_number', 'citation_category'],
                      dtype=str, chunksize=5_000_000, on_bad_lines='skip'):
    ch = ch.dropna(subset=['patent_id', 'citation_document_number'])
    cat = ch['citation_category'].fillna('').str.lower()
    notp = ~cat.str.contains('third party', na=False)
    ch = ch.loc[notp]; cat = cat.loc[notp]
    cited = ch['citation_document_number'].map(pgpub2pid)
    cy = ch['patent_id'].map(pid2year); dy = cited.map(pid2year)
    keep = (cited.notna() & cy.notna() & dy.notna()).values
    if keep.any():
        diff = (cy.values - dy.values)[keep]
        cc = cited.values[keep]
        bucket = np.where(cat.values[keep] == 'cited by examiner', 'examiner', 'non_examiner')
        pos = diff >= 0
        d = pd.DataFrame({'cited': cc[pos], 'cite_year': cy.values[keep][pos].astype(int), 'bucket': bucket[pos]})
        accA.append(d.groupby(['cited', 'cite_year', 'bucket']).size().reset_index(name='n'))
    seen += len(ch)
patapp = pd.concat(accA, ignore_index=True).groupby(['cited', 'cite_year', 'bucket'])['n'].sum().reset_index()
del pgpub2pid; gc.collect()
print(f'[{time.time()-t0:.0f}s] scanned {seen:,} app rows -> {len(patapp):,} (cited, year, bucket) groups')

crosswalk pgpub->patent: 5,432,615


[409s] scanned 78,557,720 app rows -> 20,098,313 (cited, year, bucket) groups
CPU times: total: 6min 54s
Wall time: 7min 1s


## 3. Assemble + save

In [5]:
%%time
# The same citation KINDS patent_citation.parquet carries, but resolved by year instead of by
# window. Windows collapse the time axis; this keeps it, so the two files answer different
# questions from the same edges and their totals must agree.
#
#   C            granted citations           = C_examiner + C_non_examiner + C_unknown
#   appC         application citations       = appC_examiner + appC_non_examiner
#
# `unknown` is the empty citation_category: pre-2001 rows carry no origin information at all.
# The old build folded it into non_examiner, which overstated applicant citations in exactly
# the years where the field is missing. It is now its own column, and the legacy
# pat2pat_non_examiner alias below reproduces the old (folded) definition so nothing that
# already reads this file changes meaning.
def pivot_long(long_df, prefix, buckets):
    p = (long_df.pivot_table(index=['cited', 'cite_year'], columns='bucket',
                             values='n', fill_value=0).reset_index())
    for b in buckets:
        if b not in p.columns:
            p[b] = 0
    p = p.rename(columns={b: f'{prefix}_{b}' for b in buckets})
    cols = [f'{prefix}_{b}' for b in buckets]
    p[prefix] = p[cols].sum(axis=1)
    return p[['cited', 'cite_year', prefix] + cols]


pg = pivot_long(pat,    'C',    ['examiner', 'non_examiner', 'unknown'])
pa = pivot_long(patapp, 'appC', ['examiner', 'non_examiner'])
allt = pg.merge(pa, on=['cited', 'cite_year'], how='outer')
VAL = [c for c in allt.columns if c not in ('cited', 'cite_year')]
for c in VAL:
    allt[c] = allt[c].fillna(0).astype('int64')

allt = allt.rename(columns={'cited': 'patent_id'})
allt['grant_year'] = allt['patent_id'].map(pid2year)
allt = allt.dropna(subset=['grant_year'])
allt['grant_year'] = allt['grant_year'].astype(int)
allt['cite_year'] = allt['cite_year'].astype(int)
allt = allt[allt['cite_year'] >= allt['grant_year']]
allt['yrs_since_grant'] = allt['cite_year'] - allt['grant_year']

# Legacy aliases. Anything already reading this file (patent_validation, ppp_validation) uses
# these names, and pat2pat_non_examiner meant "not examiner" INCLUDING the unknowns -- so it
# is reproduced as the sum, not as C_non_examiner alone.
allt['pat2pat_examiner']     = allt['C_examiner']
allt['pat2pat_non_examiner'] = allt['C_non_examiner'] + allt['C_unknown']
allt['app2pat_examiner']     = allt['appC_examiner']
allt['app2pat_non_examiner'] = allt['appC_non_examiner']

KINDS = ['C', 'C_examiner', 'C_non_examiner', 'C_unknown',
         'appC', 'appC_examiner', 'appC_non_examiner']
LEGACY = ['pat2pat_examiner', 'pat2pat_non_examiner', 'app2pat_examiner', 'app2pat_non_examiner']
allt = allt[['patent_id', 'grant_year', 'cite_year', 'yrs_since_grant'] + KINDS + LEGACY]
allt.to_parquet(OUT_FP, index=False, compression='zstd')
print(f'WROTE {OUT_FP}  ({len(allt):,} rows, {len(allt.columns)} cols, '
      f'{os.path.getsize(OUT_FP)/1e6:.0f} MB)')
print(f'  patents covered: {allt.patent_id.nunique():,} | '
      f'cite_year {allt.cite_year.min()}-{allt.cite_year.max()} | '
      f'yrs_since_grant 0..{allt.yrs_since_grant.max()}')

print(f'\n{"kind":<20}{"total":>16}{"share of its source":>22}')
print('-' * 58)
for src, cols in (('C', ['C_examiner', 'C_non_examiner', 'C_unknown']),
                  ('appC', ['appC_examiner', 'appC_non_examiner'])):
    tot = allt[src].sum()
    print(f'{src:<20}{tot:>16,}')
    for c in cols:
        v = allt[c].sum()
        print(f'  {c:<18}{v:>16,}{v / max(tot, 1) * 100:>21.2f}%')
assert (allt['C'] == allt[['C_examiner', 'C_non_examiner', 'C_unknown']].sum(axis=1)).all()
assert (allt['appC'] == allt[['appC_examiner', 'appC_non_examiner']].sum(axis=1)).all()
print('\n  components sum to their totals on every row')
display(allt.head(8))

WROTE C:\Users\jdwoo\OneDrive\Desktop\Research\Science of Science\notebook\patent\output\patent_citation_trend.parquet  (50,546,229 rows)


  patents covered: 7,160,531
  granted p2p total: 119,291,183 (examiner 17.1%)
  application p2p total: 49,541,000 (examiner 27.2%)


bucket,patent_id,grant_year,cite_year,yrs_since_grant,pat2pat_examiner,pat2pat_non_examiner,app2pat_examiner,app2pat_non_examiner
0,10000000,2018,2018,0,0,0,1,2
1,10000000,2018,2019,1,0,0,0,2
2,10000000,2018,2020,2,1,2,2,0
3,10000000,2018,2021,3,1,0,0,0
4,10000000,2018,2022,4,1,3,1,0
5,10000000,2018,2023,5,0,2,1,3
6,10000000,2018,2024,6,0,3,0,5
7,10000000,2018,2025,7,0,6,0,4


CPU times: total: 1min 25s
Wall time: 1min 27s
